## few shot classification

In [29]:
import vit_prisma
from vit_prisma.utils.data_utils.imagenet.imagenet_dict import IMAGENET_DICT
from vit_prisma.utils import prisma_utils

import numpy as np
import torch
from fancy_einsum import einsum
from collections import defaultdict

import plotly.graph_objs as go
import plotly.express as px

import matplotlib.colors as mcolors

from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

from IPython.core.display import display, HTML


# Helper function (ignore)
def plot_image(image):
  plt.figure()
  plt.axis('off')
  plt.imshow(image.permute(1,2,0))

class ConvertTo3Channels:
    def __call__(self, img):
        if img.mode != 'RGB':
            return img.convert('RGB')
        return img

transform = transforms.Compose([
    ConvertTo3Channels(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def plot_logit_boxplot(average_logits, labels):
  hovertexts = np.array([[IMAGENET_DICT[i] for _ in range(25)] for i in range(1000)])

  fig = go.Figure()
  data = []

  # if tensor, turn to numpy
  if isinstance(average_logits, torch.Tensor):
      average_logits = average_logits.detach().cpu().numpy()

  for i in range(average_logits.shape[1]):  # For each layer
      layer_logits = average_logits[:, i]
      hovertext = hovertexts[:, i]
      box = fig.add_trace(go.Box(
          y=layer_logits,
          name=f'{layer_labels[i]}',
          text=hovertext,
          hoverinfo='y+text',
          boxpoints='suspectedoutliers'
      ))
      data.append(box)


  means = np.mean(average_logits, axis=0)
  fig.add_trace(go.Scatter(
      x = layer_labels,
      y=means,
      mode='markers',
      name='Mean',
      # line=dict(color='gray'),
      marker=dict(size=4, color='red'),
  ))


  fig.update_layout(
      title='Raw Logit Values Per Layer (each dot is 1 ImageNet Class)',
      xaxis=dict(title='Layer'),
      yaxis=dict(title='Logit Values'),
      showlegend=False
  )

  fig.show()


def plot_patched_component(patched_head, title=''):
  """
  Use for plotting Activation Patching.
  """

  fig = go.Figure(data=go.Heatmap(
      z=patched_head.detach().numpy(),
      colorscale='RdBu',  # You can choose any colorscale
      colorbar=dict(title='Value'),  # Customize the color bar
      hoverongaps=False
  ))
  fig.update_layout(
      title=title,
      xaxis_title='Attention Head',
      yaxis_title='Patch Number',
  )

  return fig

def imshow(tensor, **kwargs):
    """
    Use for Activation Patching.
    """
    px.imshow(
          prisma_utils.to_numpy(tensor),
          color_continuous_midpoint=0.0,
          color_continuous_scale="RdBu",
          **kwargs,
      ).show()


/tmp/ipykernel_1650582/3488392042.py:19: DeprecationWarning:

Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display



In [30]:
from vit_prisma.models.base_vit import HookedViT
model = HookedViT.from_pretrained("vit_base_patch32_224",
                                        center_writing_weights=True,
                                        center_unembed=True,
                                        fold_ln=True,
                                        refactor_factored_attn_matrices=True,
                                        device="cuda"
                                    )

# Force the base model to GPU (just in case)
model = model.to("cuda:0")

# FORCE the Prisma/TransformerLens config to match
model.cfg.device = "cuda:0" 

print("Double check - Model device config:", model.cfg.device)

print("Is CUDA available to PyTorch?:", torch.cuda.is_available())

# check if model is on gpu
# assert next(model.parameters()).is_cuda, "Model is not on GPU!"
next(model.parameters()).is_cuda

2026-04-11 00:09:21 WARNING:root: Model 'vit_base_patch32_224' is not in the lists of models passing or failing tests. Unclear status. You may want to check that the HookedViT matches the original model under tests/test_loading_clip.py.
/home/nfm/prisma/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.

2026-04-11 00:09:23 DEBUG:urllib3.connectionpool: Resetting dropped connection: huggingface.co
2026-04-11 00:09:23 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /timm/vit_base_patch32_224.augreg_in21k_ft_in1k/resolve/main/config.json HTTP/1.1" 307 0
2026-04-11 00:09:23 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/timm/vit_base_patch32_224.augreg_in21k_ft_in1k/ddee2f845b54149823c7195f11246d3ab33f5285/config.json HTTP/1.1" 200

ln_pre not set


2026-04-11 00:09:25 INFO:timm.models._builder: Loading pretrained weights from Hugging Face hub (timm/vit_base_patch32_224.augreg_in21k_ft_in1k)
2026-04-11 00:09:25 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /timm/vit_base_patch32_224.augreg_in21k_ft_in1k/resolve/main/model.safetensors HTTP/1.1" 302 0
2026-04-11 00:09:25 INFO:timm.models._hub: [timm/vit_base_patch32_224.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


Converting the weights of a timm model to a Prisma ViT
LayerNorm folded.
Centered weights writing to residual stream


2026-04-11 00:09:27 INFO:root: Loaded pretrained model vit_base_patch32_224 into HookedTransformer


Double check - Model device config: cuda:0
Is CUDA available to PyTorch?: True


True

In [31]:
import os
import json
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

JSONL_PATH = "/home/nfm/adaptations/train.jsonl" # Update this to your jsonl file

unique_labels = set()
with open(JSONL_PATH, 'r') as f:
    for line in f:
        data = json.loads(line)
        unique_labels.add(data['label'])

# Sort alphabetically to guarantee artist=0, waiter=29
sorted_labels = sorted(list(unique_labels))
label_to_idx = {label: idx for idx, label in enumerate(sorted_labels)}
idx_to_label = {idx: label for idx, label in enumerate(sorted_labels)}

print(f"Found {len(sorted_labels)} classes. Mapping Example:")
print(f"Index 0: {idx_to_label[0]} | Index 29: {idx_to_label[len(sorted_labels)-1]}")

hf_dataset = load_dataset('json', data_files=JSONL_PATH, split='train')

# CRITICAL: Sort the dataset by label alphabetically!
# This guarantees that the first 20 rows are Class 0, the next 20 are Class 1, etc.
# so your batch size of 20 will perfectly group one class per batch.
hf_dataset = hf_dataset.sort('label')

# Define the transform mapping function
def transform_batch(examples):
    # 'transform' should be your standard torchvision ViT transforms (Resize, CenterCrop, Normalize)
    images = [transform(Image.open(path).convert('RGB')) for path in examples['image']]
    labels = [label_to_idx[lbl] for lbl in examples['label']]
    return {'image': images, 'label': labels}

# Apply transforms on-the-fly
hf_dataset.set_transform(transform_batch)

2026-04-11 00:10:07 DEBUG:urllib3.connectionpool: Resetting dropped connection: s3.amazonaws.com


Found 30 classes. Mapping Example:
Index 0: artist | Index 29: waiter/waitress


2026-04-11 00:10:08 DEBUG:urllib3.connectionpool: https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/datasets/json/json.py HTTP/1.1" 200 0
2026-04-11 00:10:08 DEBUG:filelock: Attempting to acquire lock 133250160158672 on /home/nfm/.cache/huggingface/datasets/_home_nfm_.cache_huggingface_datasets_json_default-6fd9b3bba2fe9bf3_0.0.0_95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2.lock
2026-04-11 00:10:08 DEBUG:filelock: Lock 133250160158672 acquired on /home/nfm/.cache/huggingface/datasets/_home_nfm_.cache_huggingface_datasets_json_default-6fd9b3bba2fe9bf3_0.0.0_95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2.lock
2026-04-11 00:10:08 DEBUG:fsspec.local: open file: /home/nfm/.cache/huggingface/datasets/json/default-6fd9b3bba2fe9bf3/0.0.0/95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2/dataset_info.json
2026-04-11 00:10:08 DEBUG:filelock: Attempting to release lock 133250160158672 on /home/nfm/.cache/huggingface/datasets/

In [33]:
# sorted_labels

In [40]:
import torch
from tqdm.auto import tqdm

# --- CONFIGURATION ---
K = 30  
DEVICE = model.cfg.device

prototypes = {}

print(f"Pre-calculating class boundaries...")

# 1. Completely strip the formatting/transforms safely
hf_dataset.reset_format()

# 2. Extract the raw string labels as a simple Python list
# (This takes a fraction of a second and bypasses image loading entirely)
all_labels = hf_dataset['label']

# 3. Find the starting index for every class
class_starts = {class_name: all_labels.index(class_name) for class_name in sorted_labels}

# 4. CRITICAL: Re-apply your image transforms so the model gets Tensors, not PIL images
hf_dataset.set_transform(transform_batch)


print(f"Generating prototypes using {K}-shot samples...")

with torch.no_grad():
    for class_name in tqdm(sorted_labels):
        
        # Get the correct start index we pre-calculated
        start_idx = class_starts[class_name]
        
        # Directly slice the K samples (your transforms are active again!)
        class_subset = hf_dataset.select(range(start_idx, start_idx + K))
        
        # Stack images into a batch
        batch_imgs = torch.stack([sample['image'] for sample in class_subset]).to(DEVICE)
        
        # Model Forward Pass
        _, cache = model.run_with_cache(batch_imgs)

        # Extract and Scale CLS token
        final_residual_stream = cache["resid_post", -1]
        cls_tokens = final_residual_stream[:, 0, :]
        scaled_tokens = cache.apply_ln_to_stack(cls_tokens, layer=-1, pos_slice=0)
        
        # Compute the Mean Prototype
        prototypes[class_name] = scaled_tokens.mean(dim=0).cpu()

print(f"Done! Created prototypes for {len(prototypes)} classes.")

# Convert to tensor and verify the standard deviations are now varied!
prototypes_tensor = torch.stack([prototypes[class_name] for class_name in sorted_labels])  
print("New Standard Deviations:", prototypes_tensor.std(dim=1))

Pre-calculating class boundaries...
Generating prototypes using 30-shot samples...


100%|██████████| 30/30 [00:06<00:00,  4.64it/s]

Done! Created prototypes for 30 classes.
New Standard Deviations: tensor([0.5957, 0.7234, 0.7780, 0.7083, 0.6776, 0.6441, 0.7260, 0.6631, 0.6968,
        0.6060, 0.6908, 0.7119, 0.6647, 0.7362, 0.6165, 0.6580, 0.7352, 0.7092,
        0.6847, 0.6925, 0.6697, 0.7307, 0.7728, 0.7323, 0.6436, 0.5793, 0.7277,
        0.6676, 0.6464, 0.6861])


In [35]:
final_residual_stream.shape  # Should be [K, 197, 768] for ViT-Base with patch size 32 and 224x224 input

torch.Size([30, 50, 768])

In [ ]:
prototypes_tensor.shape  # Should be [1000, 768]

torch.Size([30, 768])

In [41]:
prototypes_tensor.std(dim=1)

tensor([0.5957, 0.7234, 0.7780, 0.7083, 0.6776, 0.6441, 0.7260, 0.6631, 0.6968,
        0.6060, 0.6908, 0.7119, 0.6647, 0.7362, 0.6165, 0.6580, 0.7352, 0.7092,
        0.6847, 0.6925, 0.6697, 0.7307, 0.7728, 0.7323, 0.6436, 0.5793, 0.7277,
        0.6676, 0.6464, 0.6861])

In [42]:
import os
import json
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

JSONL_PATH = "/home/nfm/adaptations/val.jsonl" # Update this to your jsonl file
BATCH_SIZE = 20 # Assuming exactly 20 images per class!
LAYERS_TO_KEEP = 4 


# --- 2. BUILD ALPHABETICAL CLASS MAPPING ---
# Read the JSONL once to extract all unique labels
unique_labels = set()
with open(JSONL_PATH, 'r') as f:
    for line in f:
        data = json.loads(line)
        unique_labels.add(data['label'])

# Sort alphabetically to guarantee artist=0, waiter=29
sorted_labels = sorted(list(unique_labels))
label_to_idx = {label: idx for idx, label in enumerate(sorted_labels)}
idx_to_label = {idx: label for idx, label in enumerate(sorted_labels)}

print(f"Found {len(sorted_labels)} classes. Mapping Example:")
print(f"Index 0: {idx_to_label[0]} | Index 29: {idx_to_label[len(sorted_labels)-1]}")

# --- 3. DATASET & DATALOADER (Using HuggingFace Datasets) ---
# Load the dataset
hf_dataset = load_dataset('json', data_files=JSONL_PATH, split='train')

# CRITICAL: Sort the dataset by label alphabetically!
# This guarantees that the first 20 rows are Class 0, the next 20 are Class 1, etc.
# so your batch size of 20 will perfectly group one class per batch.
hf_dataset = hf_dataset.sort('label')

# Define the transform mapping function
def transform_batch(examples):
    # 'transform' should be your standard torchvision ViT transforms (Resize, CenterCrop, Normalize)
    images = [transform(Image.open(path).convert('RGB')) for path in examples['image']]
    labels = [label_to_idx[lbl] for lbl in examples['label']]
    return {'image': images, 'label': labels}

# Apply transforms on-the-fly
hf_dataset.set_transform(transform_batch)

2026-04-11 00:22:29 DEBUG:urllib3.connectionpool: Resetting dropped connection: s3.amazonaws.com


Found 30 classes. Mapping Example:
Index 0: artist | Index 29: waiter/waitress


2026-04-11 00:22:29 DEBUG:urllib3.connectionpool: https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/datasets/json/json.py HTTP/1.1" 200 0
2026-04-11 00:22:29 DEBUG:filelock: Attempting to acquire lock 133250288875264 on /home/nfm/.cache/huggingface/datasets/_home_nfm_.cache_huggingface_datasets_json_default-3913dce04d758245_0.0.0_95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2.lock
2026-04-11 00:22:29 DEBUG:filelock: Lock 133250288875264 acquired on /home/nfm/.cache/huggingface/datasets/_home_nfm_.cache_huggingface_datasets_json_default-3913dce04d758245_0.0.0_95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2.lock
2026-04-11 00:22:29 DEBUG:fsspec.local: open file: /home/nfm/.cache/huggingface/datasets/json/default-3913dce04d758245/0.0.0/95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2/dataset_info.json
2026-04-11 00:22:29 DEBUG:filelock: Attempting to release lock 133250288875264 on /home/nfm/.cache/huggingface/datasets/

In [43]:
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DEVICE = model.cfg.device

# 1. PRE-PROCESS PROTOTYPES
# Ensure prototypes_tensor is on GPU and L2-Normalized for Cosine Similarity
# If prototypes_tensor shape is [30, 768]
prototypes_norm = torch.nn.functional.normalize(prototypes_tensor.to(DEVICE), p=2, dim=1)

# 2. USE DATALOADER (Batching is much faster)
val_dataloader = DataLoader(hf_dataset, batch_size=32, shuffle=False)

results = []

with torch.no_grad():
    for batch in tqdm(val_dataloader):
        batch_imgs = batch['image'].to(DEVICE)
        true_labels = batch['label'] # These are already indices if using your previous map
        
        # Forward pass
        _, cache = model.run_with_cache(batch_imgs)
        
        # Extract and scale CLS tokens
        # Shape: [Batch, Seq, Dim] -> [Batch, Dim]
        cls_tokens = cache["resid_post", -1][:, 0, :]
        scaled_tokens = cache.apply_ln_to_stack(cls_tokens, layer=-1, pos_slice=0)
        
        # L2-Normalize the query embeddings
        query_norm = torch.nn.functional.normalize(scaled_tokens, p=2, dim=1)
        
        # 3. VECTORIZED SIMILARITY (Matrix Multiplication)
        # [Batch, 768] @ [768, 30] -> [Batch, 30]
        # Since both are normalized, this IS the cosine similarity.
        scores = query_norm @ prototypes_norm.T
        
        # Get predictions
        predicted_indices = torch.argmax(scores, dim=1)
        
        # Store or Print
        for i in range(len(true_labels)):
            t_idx = true_labels[i].item()
            p_idx = predicted_indices[i].item()
            similarity = scores[i, p_idx].item()
            
            print(f"True: {idx_to_label[t_idx]} | Pred: {idx_to_label[p_idx]} | Sim: {similarity:.4f}")
            results.append((t_idx, p_idx))

# Optional: Calculate Accuracy
correct = sum(1 for t, p in results if t == p)
print(f"\nFinal Accuracy: {100 * correct / len(results):.2f}%")

  5%|▌         | 1/19 [00:00<00:04,  3.83it/s]

True: artist | Pred: artist | Sim: 0.6662
True: artist | Pred: laborer/worker | Sim: 0.6706
True: artist | Pred: vendor/salesperson | Sim: 0.8074
True: artist | Pred: laborer/worker | Sim: 0.3458
True: artist | Pred: student | Sim: 0.6121
True: artist | Pred: photographer/cameraman | Sim: 0.6511
True: artist | Pred: skateboarder | Sim: 0.6747
True: artist | Pred: skateboarder | Sim: 0.6654
True: artist | Pred: tourist | Sim: 0.6708
True: artist | Pred: artist | Sim: 0.2601
True: artist | Pred: student | Sim: 0.5429
True: artist | Pred: artist | Sim: 0.6034
True: artist | Pred: surfer | Sim: 0.5324
True: artist | Pred: tourist | Sim: 0.6999
True: artist | Pred: artist | Sim: 0.6445
True: artist | Pred: artist | Sim: 0.6411
True: artist | Pred: photographer/cameraman | Sim: 0.7212
True: artist | Pred: student | Sim: 0.6568
True: artist | Pred: tourist | Sim: 0.7102
True: artist | Pred: tourist | Sim: 0.6663
True: barber | Pred: barber | Sim: 0.6813
True: barber | Pred: barber | Sim: 0.48

 11%|█         | 2/19 [00:00<00:04,  4.13it/s]

True: barber | Pred: waiter/waitress | Sim: 0.7522
True: barber | Pred: artist | Sim: 0.3452
True: barber | Pred: barber | Sim: 0.8658
True: barber | Pred: barber | Sim: 0.6586
True: barber | Pred: barber | Sim: 0.8050
True: barber | Pred: barber | Sim: 0.7689
True: barber | Pred: barber | Sim: 0.8616
True: barber | Pred: vendor/salesperson | Sim: 0.7262
True: baseball player | Pred: baseball player | Sim: 0.7748
True: baseball player | Pred: baseball player | Sim: 0.8039
True: baseball player | Pred: sports referee | Sim: 0.8603
True: baseball player | Pred: sports referee | Sim: 0.8374
True: baseball player | Pred: baseball player | Sim: 0.8156
True: baseball player | Pred: baseball player | Sim: 0.8120
True: baseball player | Pred: baseball player | Sim: 0.6279
True: baseball player | Pred: baseball player | Sim: 0.8100
True: baseball player | Pred: baseball player | Sim: 0.8775
True: baseball player | Pred: sports referee | Sim: 0.6837
True: baseball player | Pred: baseball player 

 21%|██        | 4/19 [00:00<00:03,  4.49it/s]

...

True: soccer player | Pred: soccer player | Sim: 0.5359
True: soccer player | Pred: soccer player | Sim: 0.8465
True: soccer player | Pred: soccer player | Sim: 0.5476
True: soccer player | Pred: soccer player | Sim: 0.8363
True: soccer player | Pred: soccer player | Sim: 0.8881
True: soccer player | Pred: soccer player | Sim: 0.8657
True: soccer player | Pred: soccer player | Sim: 0.8652
True: soccer player | Pred: soccer player | Sim: 0.7858
True: soccer player | Pred: soccer player | Sim: 0.6464
True: soccer player | Pred: businessman/businesswoman | Sim: 0.7495
True: soccer player | Pred: soccer player | Sim: 0.7767
True: soccer player | Pred: soccer player | Sim: 0.8150
True: sports referee | Pred: tennis player | Sim: 0.6114
True: sports referee | Pred: sports referee | Sim: 0.5306
True: sports referee | Pred: sports referee | Sim: 0.6775
True: sports referee | Pred: baseball player | Sim: 0.3433
True: sports referee | Pred: tennis player | Sim: 0.6597
True: sports referee | Pred

 89%|████████▉ | 17/19 [00:03<00:00,  4.72it/s]

True: surfer | Pred: tourist | Sim: 0.7474
True: surfer | Pred: surfer | Sim: 0.7393
True: surfer | Pred: surfer | Sim: 0.7852
True: surfer | Pred: surfer | Sim: 0.6039
True: surfer | Pred: surfer | Sim: 0.6196
True: surfer | Pred: surfer | Sim: 0.6876
True: surfer | Pred: surfer | Sim: 0.5656
True: surfer | Pred: surfer | Sim: 0.5366
True: tennis player | Pred: tennis player | Sim: 0.8455
True: tennis player | Pred: tennis player | Sim: 0.8779
True: tennis player | Pred: tennis player | Sim: 0.7286
True: tennis player | Pred: tennis player | Sim: 0.5660
True: tennis player | Pred: tennis player | Sim: 0.8221
True: tennis player | Pred: tennis player | Sim: 0.8162
True: tennis player | Pred: tennis player | Sim: 0.8418
True: tennis player | Pred: tennis player | Sim: 0.8272
True: tennis player | Pred: tennis player | Sim: 0.8200
True: tennis player | Pred: tennis player | Sim: 0.7785
True: tennis player | Pred: tennis player | Sim: 0.8819
True: tennis player | Pred: tennis player | Sim

100%|██████████| 19/19 [00:04<00:00,  4.54it/s]

True: tourist | Pred: frisbee player | Sim: 0.7085
True: tourist | Pred: police officer | Sim: 0.6189
True: tourist | Pred: tourist | Sim: 0.6195
True: tourist | Pred: tourist | Sim: 0.7001
True: tourist | Pred: tourist | Sim: 0.6047
True: tourist | Pred: surfer | Sim: 0.6517
True: tourist | Pred: cyclist | Sim: 0.4168
True: tourist | Pred: tourist | Sim: 0.6542
True: tourist | Pred: tourist | Sim: 0.6884
True: tourist | Pred: tourist | Sim: 0.7360
True: tourist | Pred: tourist | Sim: 0.7453
True: tourist | Pred: tourist | Sim: 0.6538
True: tourist | Pred: skier | Sim: 0.2933
True: tourist | Pred: skateboarder | Sim: 0.5184
True: tourist | Pred: photographer/cameraman | Sim: 0.4332
True: tourist | Pred: tourist | Sim: 0.7171
True: vendor/salesperson | Pred: vendor/salesperson | Sim: 0.7493
True: vendor/salesperson | Pred: musician/instrumentalist | Sim: 0.6060
True: vendor/salesperson | Pred: vendor/salesperson | Sim: 0.5938
True: vendor/salesperson | Pred: vendor/salesperson | Sim: 0.

In [2]:
import numpy as np
pth = "/home/nfm/clip_text_span/output_dir/xxx_cls_attn_ViT-B-16.npy"

pth2 = "/home/nfm/clip_text_span/output_dir/xxx_attn_ViT-B-16.npy"

# cls_attn = np.load(pth)
# cls_attn.shape

# loas very large attention map with 14gb of memory, so we need to use numpy memmap
cls_attn_memmap = np.load(pth2, mmap_mode='r')
cls_attn_memmap.shape

(50000, 12, 12, 512)

In [5]:
import json
LOCAL_JSON_PATH = "/home/nfm/ViT-Prisma/demos/imagenet_class_index.json"
with open(LOCAL_JSON_PATH, 'r') as f:
    imagenet_class_index = json.load(f)

wnid_to_name = {}
for idx, (wnid, class_name) in imagenet_class_index.items():
    safe_class_name = class_name.replace(" ", "_").replace("/", "_").replace(",", "")
    wnid_to_name[wnid] = safe_class_name

wnid_to_idx = {wnid: int(idx) for idx, (wnid, name) in imagenet_class_index.items()}
idx_to_name = {int(idx): name for idx, (wnid, name) in imagenet_class_index.items()}

idx_to_wnid = {int(idx): wnid for idx, (wnid, name) in imagenet_class_index.items()}
name_to_idx = {name: int(idx) for idx, (wnid, name) in imagenet_class_index.items()}

In [8]:
from urllib.request import urlopen
from PIL import Image
import timm
import torch

img = Image.open(urlopen(
    'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png'
))

model = timm.create_model('vit_base_patch32_clip_224.laion2b_ft_in1k', pretrained=True)
model = model.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

output = model(transforms(img).unsqueeze(0))  # unsqueeze single image into batch of 1

top5_probabilities, top5_class_indices = torch.topk(output.softmax(dim=1) * 100, k=5)
# print("Top 5 Predictions:")
for prob, idx in zip(top5_probabilities[0], top5_class_indices[0]):
    class_name = idx_to_name[idx.item()]
    print(f"{class_name}: {prob.item():.2f}%")

espresso: 46.34%
coffee_mug: 11.87%
cup: 9.77%
chocolate_sauce: 9.05%
coffeepot: 1.84%


In [9]:
model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximat